In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

required = [
    "seriea_1516_events.pkl", "premier_1516_events.pkl", "laliga_1516_events.pkl", "ligue1_1516_events.pkl",
    "all_1516_merged.pkl", "all_1516_dims.pkl", "all_1516_scored_v2.pkl", "all_1516_backtest.pkl",
    "team_possession_1516.pkl", "transfermarkt-datasets.duckdb", "oos_candidates.csv",
]
for f in required:
    p = f"{folder}/{f}"
    print(("OK      " if os.path.exists(p) else "НЕТ     ") + f,
          f"({os.path.getsize(p) / 1e6:.0f} МБ)" if os.path.exists(p) else "")

bt_check = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
need_cols = ["score", "role", "league", "team", "age", "mins", "value_eur", "v12", "v24", "v36",
             "d12", "d24", "d36", "logv", "tm_name"]
print("\nнедостающие колонки в all_1516_backtest.pkl:", [c for c in need_cols if c not in bt_check.columns])
print("игроков:", len(bt_check))

Mounted at /content/drive
OK      seriea_1516_events.pkl (380 МБ)
OK      premier_1516_events.pkl (372 МБ)
OK      laliga_1516_events.pkl (374 МБ)
OK      ligue1_1516_events.pkl (384 МБ)
OK      all_1516_merged.pkl (1 МБ)
OK      all_1516_dims.pkl (1 МБ)
OK      all_1516_scored_v2.pkl (1 МБ)
OK      all_1516_backtest.pkl (1 МБ)
OK      team_possession_1516.pkl (0 МБ)
OK      transfermarkt-datasets.duckdb (211 МБ)
OK      oos_candidates.csv (0 МБ)

недостающие колонки в all_1516_backtest.pkl: []
игроков: 1232


In [2]:
bt = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
for c in ["score", "age", "mins", "value_eur", "v12", "v24", "v36", "d12", "d24", "d36", "logv"]:
    bt[c] = pd.to_numeric(bt[c], errors="coerce")
bt = bt[bt.mins >= 1500].dropna(subset=["score", "age", "value_eur"]).copy()

group_of = {"ST": "ATT", "AM_W": "ATT", "CM": "MID", "DM": "MID", "FB": "DEF", "CB": "DEF"}
bt["group"] = bt.role.map(group_of)
bt["spct"] = bt.groupby("role").score.rank(pct=True)
bt["name"] = [t if isinstance(t, str) else i for t, i in zip(bt.tm_name, bt.index)]

def mm(x):
    return round(x / 1e6, 1) if pd.notna(x) else np.nan

tops, cheap = {}, {}
for g in ["ATT", "MID", "DEF"]:
    s = bt[bt.group == g]
    top = s.sort_values("spct", ascending=False).head(50)
    tops[g] = top
    cheap[g] = top.sort_values("value_eur").head(15)
    out = pd.DataFrame({"player": top.name, "team": top.team, "league": top.league, "role": top.role,
                        "age": top.age.round(1), "mins": top.mins.round(), "score": top.score.round(0),
                        "value_2016_m": top.value_eur.map(mm), "2017": top.v12.map(mm),
                        "2018": top.v24.map(mm), "2019": top.v36.map(mm),
                        "cheap": top.index.isin(cheap[g].index)})
    out.to_csv(f"{folder}/top50_{g}.csv", index=False)

    print(f"\n=== {g}: лучшие 10 по оценке (проверка здравого смысла) ===")
    print("; ".join(f"{r['name']} ({r.team}, {r.value_eur / 1e6:.0f}M)" for _, r in top.head(10).iterrows()))
    print(f"\n=== {g}: 15 самых дешёвых из топ-50 (стоимость в млн евро: 2016 / 2017 / 2018 / 2019) ===")
    c = cheap[g].sort_values("value_eur")
    print(pd.DataFrame({"player": c.name, "team": c.team, "role": c.role, "age": c.age.round(0),
                        "score": c.score.round(0), "2016": c.value_eur.map(mm), "2017": c.v12.map(mm),
                        "2018": c.v24.map(mm), "2019": c.v36.map(mm)}).to_string(index=False))

def pct(b):
    return (np.exp(b) - 1) * 100

print("\n=== ПРОВЕРКА: дешёвые из топ-50 против других игроков в том же ценовом диапазоне ===")
pool_parts = []
for g in ["ATT", "MID", "DEF"]:
    s = bt[bt.group == g].copy()
    s = s[s.value_eur <= cheap[g].value_eur.max()]
    s["top"] = s.index.isin(tops[g].index).astype(int)
    pool_parts.append(s)
pool = pd.concat(pool_parts)
print("игроков в ценовых диапазонах:", len(pool), "| из них из топ-50:", int(pool.top.sum()))
print("средний возраст: топ-50", round(pool[pool.top == 1].age.mean(), 1),
      "| остальные", round(pool[pool.top == 0].age.mean(), 1))

for h in (12, 24, 36):
    col = f"d{h}"
    s = pool[pool[col].notna()]
    m = smf.ols(f"{col} ~ top + logv + age + I(age**2) + C(role) + C(league)", data=s).fit(cov_type="HC3")
    lo, hi = m.conf_int().loc["top"].values
    t, o = s[s.top == 1], s[s.top == 0]
    print(f"{h} мес: топ-50 n={len(t)} vs другие n={len(o)} | сырая медиана {pct(t[col].median()):+.0f}% vs "
          f"{pct(o[col].median()):+.0f}% | выросли {(t[col] > 0).mean() * 100:.0f}% vs {(o[col] > 0).mean() * 100:.0f}% "
          f"| с поправкой на цену, возраст, роль: {pct(m.params['top']):+.0f}% "
          f"(95% ДИ {pct(lo):+.0f}% … {pct(hi):+.0f}%), p={m.pvalues['top']:.3f}")

print("\nпо группам, 24 месяца (сырые медианы, небольшие выборки):")
for g in ["ATT", "MID", "DEF"]:
    s = pool[(pool.group == g) & pool.d24.notna()]
    t, o = s[s.top == 1], s[s.top == 0]
    print(f"{g}: топ-50 n={len(t)} медиана {pct(t.d24.median()):+.0f}% | другие n={len(o)} медиана {pct(o.d24.median()):+.0f}%")
print("\nфайлы сохранены: top50_ATT.csv, top50_MID.csv, top50_DEF.csv")


=== ATT: лучшие 10 по оценке (проверка здравого смысла) ===
Paulo Dybala (Juventus, 40M); Lionel Messi (Barcelona, 120M); Alexis Sánchez (Arsenal, 55M); Sergio Agüero (Manchester City, 60M); Neymar (Barcelona, 100M); Karim Benzema (Real Madrid, 60M); Nolito (Celta Vigo, 15M); Rachid Ghezzal (Lyon, 2M); Stevan Jovetić (Inter Milan, 15M); Philippe Coutinho (Liverpool, 32M)

=== ATT: 15 самых дешёвых из топ-50 (стоимость в млн евро: 2016 / 2017 / 2018 / 2019) ===
            player                   team role  age  score  2016  2017  2018  2019
    Rachid Ghezzal                   Lyon AM_W 24.0   83.0   2.5   8.0   4.0   8.0
        Miku Fedor         Rayo Vallecano   ST 31.0   71.0   2.5   1.0   NaN   0.5
      Aritz Aduriz          Athletic Club   ST 35.0   70.0   3.0   1.0   1.0   1.0
    Jonathan Viera             Las Palmas AM_W 27.0   76.0   3.5  10.0   8.0   NaN
   Fabián Orellana             Celta Vigo AM_W 30.0   68.0   4.0   3.0   2.0   2.5
       Andy Delort    Stade Malherbe

In [4]:
!pip install statsbombpy

import os, warnings
import pandas as pd
from statsbombpy import sb
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
os.makedirs(f"{folder}/parts2", exist_ok=True)

season = "2015/2016"
leagues = {"seriea": "Serie A", "premier": "Premier League", "laliga": "La Liga", "ligue1": "Ligue 1"}

keep2 = [
    "id", "match_id", "team", "player", "player_id", "position", "type", "period", "minute", "second",
    "location", "under_pressure", "possession", "possession_team",
    "pass_end_location", "pass_outcome", "pass_length", "pass_height", "pass_body_part", "pass_type",
    "pass_cross", "pass_through_ball", "pass_switch", "pass_shot_assist", "pass_goal_assist",
    "pass_assisted_shot_id", "pass_aerial_won",
    "shot_outcome", "shot_statsbomb_xg", "shot_type", "shot_body_part", "shot_key_pass_id",
    "shot_end_location", "shot_aerial_won",
    "dribble_outcome", "duel_type", "duel_outcome", "interception_outcome",
    "clearance_aerial_won", "miscontrol_aerial_won", "counterpress", "carry_end_location",
    "goalkeeper_type", "goalkeeper_outcome", "goalkeeper_position",
    "foul_committed_card", "tactics", "substitution_replacement",
]

comps = sb.competitions()

def load_league(key, name, chunk=100):
    row = comps[(comps.competition_name == name) & (comps.season_name == season)].iloc[0]
    matches = sb.matches(competition_id=row.competition_id, season_id=row.season_id)
    ids = list(matches.match_id)
    rows, cols = 0, set()
    for start in range(0, len(ids), chunk):
        part_path = f"{folder}/parts2/{key}_{start}.pkl"
        if os.path.exists(part_path):
            df = pd.read_pickle(part_path)
        else:
            frames = []
            for mid in ids[start:start + chunk]:
                try:
                    e = sb.events(match_id=mid)
                except Exception as ex:
                    print("ошибка в матче", mid, ex)
                    continue
                frames.append(e[[c for c in keep2 if c in e.columns]])
            df = pd.concat(frames, ignore_index=True)
            df.to_pickle(part_path)
            print(name, min(start + chunk, len(ids)), "из", len(ids))
        rows += len(df)
        cols |= set(df.columns)
        del df
    return rows, [c for c in keep2 if c not in cols]

for key, name in leagues.items():
    rows, missing = load_league(key, name)
    print(f"{name}: строк {rows} | колонок из списка нет в данных: {missing}")
print("готово")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Serie A 100 из 380
Serie A 200 из 380
Serie A 300 из 380
Serie A 380 из 380
Serie A: строк 1353739 | колонок из списка нет в данных: []
Premier League 100 из 380
Premier League 200 из 380
Premier League 300 из 380
Premier League 380 из 380
Premier League: строк 1313773 | колонок из списка нет в данных: []
La Liga 100 из 380
La Liga 200 из 380
La Liga 300 из 380
La Liga 380 из 380
La Liga: строк 1295354 | колонок из списка нет в данных: []
Ligue 1 100 из 377
Ligue 1 200 из 377
Ligue 1 300 из 377
Ligue 1 377 из 377
Ligue 1: строк 1358593 | колонок из списка нет в данных: []
готово


In [5]:
import glob, re
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

leagues = {"seriea": "Serie A", "premier": "Premier League", "laliga": "La Liga", "ligue1": "Ligue 1"}
poss = pd.read_pickle(f"{folder}/team_possession_1516.pkl")
base = pd.read_pickle(f"{folder}/all_1516_merged.pkl")[["league", "team", "role", "mins", "age", "tm_id", "tm_name"]]
base = base.rename_axis("player").reset_index()
base["age"] = pd.to_numeric(base.age, errors="coerce")

def load_events(key):
    files = sorted(glob.glob(f"{folder}/parts2/{key}_*.pkl"),
                   key=lambda p: int(re.search(r"_(\d+)\.pkl$", p).group(1)))
    return pd.concat([pd.read_pickle(p) for p in files], ignore_index=True)

def box(x, y):
    return (x >= 102) & (y >= 18) & (y <= 62)

def league_stats(key, name):
    ev = load_events(key)

    def xy(col, prefix):
        ev[prefix + "_x"] = np.nan
        ev[prefix + "_y"] = np.nan
        s = ev[col]
        ok = s.notna()
        arr = pd.DataFrame(s[ok].tolist(), index=s[ok].index)
        ev.loc[ok, prefix + "_x"] = arr[0].values
        ev.loc[ok, prefix + "_y"] = arr[1].values

    xy("location", "s")
    xy("pass_end_location", "pe")
    xy("carry_end_location", "ce")
    dg = lambda x, y: np.sqrt((120 - x) ** 2 + (40 - y) ** 2)
    ev["pass_prog"] = dg(ev.s_x, ev.s_y) - dg(ev.pe_x, ev.pe_y)
    ev["carry_prog"] = dg(ev.s_x, ev.s_y) - dg(ev.ce_x, ev.ce_y)

    t = ev.type
    ispass = t == "Pass"
    done = ev.pass_outcome.isna()
    press = ev.under_pressure == True
    isshot = t == "Shot"
    pen = ev.shot_type == "Penalty"
    tackle = (t == "Duel") & (ev.duel_type == "Tackle")
    won_out = ["Won", "Success In Play", "Success Out"]
    aerial_won = ((ev.pass_aerial_won == True) | (ev.clearance_aerial_won == True)
                  | (ev.shot_aerial_won == True) | (ev.miscontrol_aerial_won == True))

    shot_xg = ev.loc[isshot, ["id", "shot_statsbomb_xg"]].set_index("id").shot_statsbomb_xg
    xa = ev.pass_assisted_shot_id.map(shot_xg).fillna(0)
    xg = ev.shot_statsbomb_xg.fillna(0)

    f = pd.DataFrame({
        "passes": ispass, "passes_done": ispass & done,
        "prog_passes": ispass & done & (ev.pass_prog >= 10),
        "press_passes": ispass & press, "press_passes_done": ispass & press & done,
        "crosses_done": ispass & done & (ev.pass_cross == True),
        "key_passes": (ev.pass_shot_assist == True) | (ev.pass_goal_assist == True),
        "np_shots": isshot & ~pen,
        "touches_box": (t == "Ball Receipt*") & box(ev.s_x, ev.s_y),
        "dribbles": t == "Dribble",
        "dribbles_won": (t == "Dribble") & (ev.dribble_outcome == "Complete"),
        "prog_carries": (t == "Carry") & (ev.carry_prog >= 10),
        "tackles": tackle, "tackles_won": tackle & ev.duel_outcome.isin(won_out),
        "aerial_won": aerial_won,
        "aerial_lost": (t == "Duel") & (ev.duel_type == "Aerial Lost"),
        "interceptions": t == "Interception", "recoveries": t == "Ball Recovery",
        "pressures": t == "Pressure", "clearances": t == "Clearance", "blocks": t == "Block",
        "fouls": t == "Foul Committed",
    })
    f["xa"] = xa
    f["npxg"] = xg.where(isshot & ~pen, 0)
    f["player"] = ev.player
    st = f[f.player.notna()].groupby("player").sum()
    st["league"] = name
    print(name, "готово")
    return st

st = pd.concat([league_stats(k, n) for k, n in leagues.items()]).reset_index()
d = base.merge(st, on=["player", "league"], how="left")
print("игроков без статистики после объединения:", int(d.passes.isna().sum()), "из", len(d))

# поправка на владение (k = 0.2, как в проекте 2)
d["poss"] = [poss.get((l, t), np.nan) for l, t in zip(d.league, d.team)]
fac = (0.5 / (1 - d.poss)) ** 0.2
for m in ["tackles", "interceptions", "recoveries", "pressures", "clearances", "blocks"]:
    d[m + "_adj"] = d[m] * fac

def p90(c):
    return d[c] / d.mins * 90

def shrink(succ, tot, k):
    prior = succ.groupby(d.role).transform("sum") / tot.groupby(d.role).transform("sum")
    return (succ + k * prior) / (tot + k)

M = pd.DataFrame(index=d.index)
M["npxg90"] = p90("npxg")
M["shots90"] = p90("np_shots")
M["xa90"] = p90("xa")
M["key_passes90"] = p90("key_passes")
M["touches_box90"] = p90("touches_box")
M["prog_passes90"] = p90("prog_passes")
M["prog_carries90"] = p90("prog_carries")
M["crosses_done90"] = p90("crosses_done")
M["dribbles_won90"] = p90("dribbles_won")
M["pressures_adj90"] = p90("pressures_adj")
M["recoveries_adj90"] = p90("recoveries_adj")
M["interceptions_adj90"] = p90("interceptions_adj")
M["clearances_adj90"] = p90("clearances_adj")
M["ti_adj90"] = (d.tackles_adj + d.interceptions_adj) / d.mins * 90
M["pass_pct"] = d.passes_done / d.passes
M["fouls_low90"] = -p90("fouls")
M["npxg_per_shot"] = shrink(d.npxg, d.np_shots, 5)
M["tackle_pct_s"] = shrink(d.tackles_won, d.tackles, 10)
M["aerial_pct_s"] = shrink(d.aerial_won, d.aerial_won + d.aerial_lost, 10)
M["pass_pct_press_s"] = shrink(d.press_passes_done, d.press_passes, 30)

SPEC = {
    "ST": ["npxg90", "shots90", "npxg_per_shot", "touches_box90", "xa90", "pressures_adj90"],
    "AM_W": ["xa90", "key_passes90", "npxg90", "touches_box90", "prog_carries90", "dribbles_won90"],
    "CM": ["prog_passes90", "pass_pct", "pass_pct_press_s", "xa90", "prog_carries90", "ti_adj90", "recoveries_adj90"],
    "DM": ["prog_passes90", "pass_pct", "pass_pct_press_s", "tackle_pct_s", "ti_adj90", "recoveries_adj90", "aerial_pct_s"],
    "FB": ["crosses_done90", "xa90", "prog_carries90", "prog_passes90", "tackle_pct_s", "ti_adj90"],
    "CB": ["aerial_pct_s", "tackle_pct_s", "interceptions_adj90", "clearances_adj90", "prog_passes90", "pass_pct", "fouls_low90"],
}
all_m = sorted({m for v in SPEC.values() for m in v})
P = pd.DataFrame(np.nan, index=d.index, columns=all_m)
d["comp"], d["n_high"] = np.nan, np.nan
for role, mets in SPEC.items():
    idx = d.role == role
    for m in mets:
        P.loc[idx, m] = M.loc[idx, m].rank(pct=True)
    d.loc[idx, "comp"] = P.loc[idx, mets].mean(axis=1)
    d.loc[idx, "n_high"] = (P.loc[idx, mets] >= 0.8).sum(axis=1)

d["group"] = d.role.map({"ST": "ATT", "AM_W": "ATT", "CM": "MID", "DM": "MID", "FB": "DEF", "CB": "DEF"})
d["name"] = [t if isinstance(t, str) else p for t, p in zip(d.tm_name, d.player)]

def top3(i):
    s = P.loc[i, SPEC[d.loc[i, "role"]]].sort_values(ascending=False).head(3)
    return ", ".join(f"{k} {v * 100:.0f}" for k, v in s.items())

def table(df):
    return pd.DataFrame({"player": df.name, "team": df.team, "role": df.role, "age": df.age.round(1),
                         "mins": df.mins.round(), "comp": (df.comp * 100).round(0),
                         "n_high": df.n_high, "best metrics (percentile)": [top3(i) for i in df.index]}
                        ).to_string(index=False)

print("\nвсего игроков по ролям (900+ минут):", d.role.value_counts().to_dict())
u = d[(d.age < 23) & (d.mins >= 1500)].copy()
print("до 23 лет и 1500+ минут:", len(u), u.role.value_counts().to_dict())
print("игроков без возраста:", int(d.age.isna().sum()))

print("\n=== ПРОВЕРКА ЗДРАВОГО СМЫСЛА: лучшие 6 по каждой роли среди ВСЕХ возрастов ===")
for role in SPEC:
    t = d[d.role == role].sort_values("comp", ascending=False).head(6)
    print(f"{role}:", "; ".join(f"{r['name']} ({r.team}, {r.comp * 100:.0f})" for _, r in t.iterrows()))

A = u[(u.comp >= 0.85) & (u.n_high >= 3)].sort_values("comp", ascending=False)
print(f"\n=== УРОВЕНЬ A: явные лидеры до 23 лет (средний процентиль ≥ 85 и ≥ 3 метрик ≥ 80): {len(A)} ===")
if len(A):
    print(table(A))

print("\n=== УРОВЕНЬ B: пять лучших до 23 лет в каждой группе ===")
tierB = []
for g in ["ATT", "MID", "DEF"]:
    t = u[u.group == g].sort_values("comp", ascending=False).head(5)
    tierB.append(t)
    print(f"\n{g}")
    print(table(t))
tierB = pd.concat(tierB)

out = pd.concat([d, M.add_prefix("m_"), P.add_prefix("p_")], axis=1)
out.to_pickle(f"{folder}/u23_scouting_all.pkl")
sel = out.loc[sorted(set(A.index) | set(tierB.index))].copy()
sel["tier"] = np.where(sel.index.isin(A.index), "A", "B")
sel.to_pickle(f"{folder}/u23_selected.pkl")
print("\nсохранено: u23_scouting_all.pkl, u23_selected.pkl | выбрано игроков:", len(sel))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Serie A готово
Premier League готово
La Liga готово
Ligue 1 готово
игроков без статистики после объединения: 0 из 1251

всего игроков по ролям (900+ минут): {'FB': 265, 'CB': 251, 'AM_W': 220, 'DM': 180, 'CM': 171, 'ST': 164}
до 23 лет и 1500+ минут: 101 {'AM_W': 25, 'FB': 21, 'CM': 21, 'CB': 16, 'ST': 11, 'DM': 7}
игроков без возраста: 19

=== ПРОВЕРКА ЗДРАВОГО СМЫСЛА: лучшие 6 по каждой роли среди ВСЕХ возрастов ===
ST: Luis Suárez (Barcelona, 81); Kévin Gameiro (Sevilla, 78); Alexandre Lacazette (Lyon, 77); Miku Fedor (Rayo Vallecano, 75); Olivier Giroud (Arsenal, 75); Christian Benteke (Liverpool, 74)
AM_W: Neymar (Barcelona, 98); Lionel Messi (Barcelona, 96); Alexis Sánchez (Arsenal, 87); Gareth Bale (Real Madrid, 84); Nolito (Celta Vigo, 84); Lorenzo Insigne (Napoli, 83)
CM: Luka Modrić (Real Madrid, 86); Marco Verratti (Paris Saint-Germain, 85); Mounir

In [1]:
import numpy as np
import pandas as pd
import duckdb
import statsmodels.formula.api as smf
from scipy.stats import fisher_exact
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
con = duckdb.connect(f"{folder}/transfermarkt-datasets.duckdb", read_only=True)

allp = pd.read_pickle(f"{folder}/u23_scouting_all.pkl")
u = allp[(allp.age < 23) & (allp.mins >= 1500)].copy()
tierB = pd.concat([u[u.group == g].sort_values("comp", ascending=False).head(5) for g in ["ATT", "MID", "DEF"]])
tier_role = pd.concat([u[u.role == r].sort_values("comp", ascending=False).head(3)
                       for r in ["ST", "AM_W", "CM", "DM", "FB", "CB"]])
u["sel"] = u.index.isin(set(tierB.index) | set(tier_role.index)).astype(int)
print("молодых игроков (до 23, 1500+ минут):", len(u), "| выбрано:", int(u.sel.sum()),
      "| без Transfermarkt-id:", int(u.tm_id.isna().sum()))
o = u[u.tm_id.notna()].copy()
o["tm_id"] = o.tm_id.astype(int)
ids = ",".join(str(i) for i in o.tm_id.unique())

pl = con.sql(f"""SELECT player_id AS tm_id, last_season, current_club_name AS club,
                        current_club_domestic_competition_id AS cur_comp, market_value_in_eur AS cur_value,
                        highest_market_value_in_eur AS peak_value, international_caps AS caps
                 FROM players WHERE player_id IN ({ids})""").df().set_index("tm_id")
val = con.sql(f"""SELECT player_id AS tm_id, date, market_value_in_eur AS v FROM player_valuations
                  WHERE player_id IN ({ids}) AND market_value_in_eur IS NOT NULL""").df()
app = con.sql(f"""SELECT player_id AS tm_id, date, competition_id AS comp, minutes_played AS mp FROM appearances
                  WHERE player_id IN ({ids}) AND competition_id IN ('GB1','ES1','IT1','FR1','L1')
                  AND date >= DATE '2016-07-01'""").df()
val["date"] = pd.to_datetime(val.date)
app["date"] = pd.to_datetime(app.date)

def at(year):
    t = pd.Timestamp(f"{year}-06-30")
    w = val[(val.date >= t - pd.Timedelta(days=150)) & (val.date <= t + pd.Timedelta(days=30))].copy()
    w["gap"] = (w.date - t).abs()
    return w.sort_values("gap").groupby("tm_id").head(1).set_index("tm_id").v

cut = pd.Timestamp("2016-06-30")
b = val[val.date <= cut].sort_values("date").groupby("tm_id").tail(1)
a = val[(val.date > cut) & ~val.tm_id.isin(b.tm_id)].sort_values("date").groupby("tm_id").head(1)
start = pd.concat([b, a]).set_index("tm_id").v
pk = val.sort_values(["tm_id", "v", "date"], ascending=[True, False, True]).groupby("tm_id").head(1).set_index("tm_id")
last = val.sort_values("date").groupby("tm_id").tail(1).set_index("tm_id")

def season_mins(y):
    s, e = pd.Timestamp(f"{y}-07-01"), pd.Timestamp(f"{y + 1}-06-30")
    return app[(app.date >= s) & (app.date <= e)].groupby("tm_id").mp.sum()

o["start"] = o.tm_id.map(start)
for y in (2017, 2018, 2019, 2021):
    o[f"v{y}"] = o.tm_id.map(at(y))
o["peak"] = o.tm_id.map(pk.v)
o["peak_year"] = o.tm_id.map(pk.date.dt.year)
o["now"] = o.tm_id.map(pl.cur_value)
o["now_date"] = o.tm_id.map(last.date.dt.year)
o["club"] = o.tm_id.map(pl.club)
o["last_season"] = o.tm_id.map(pl.last_season)
o["caps"] = o.tm_id.map(pl.caps).fillna(0)
o["m1819"] = o.tm_id.map(season_mins(2018)).fillna(0)
o["m1920"] = o.tm_id.map(season_mins(2019)).fillna(0)
o["top5_late"] = ((o.m1819 >= 900) | (o.m1920 >= 900)).astype(int)
o["g19"] = np.log(o.v2019 / o.start)
M = lambda x: np.round(x / 1e6, 1)

print("\n=== ЧТО СТАЛО С ВЫБРАННЫМИ (млн евро; top5 = минуты в пяти топ-лигах в 2018/19 и 2019/20) ===")
s = o[o.sel == 1].sort_values(["role", "comp"], ascending=[True, False])
print(pd.DataFrame({
    "player": s.name, "team": s.team, "role": s.role, "age": s.age.round(0),
    "comp": (s.comp * 100).round(0), "2016": M(s.start), "2017": M(s.v2017), "2018": M(s.v2018),
    "2019": M(s.v2019), "2021": M(s.v2021), "peak": M(s.peak), "peak_y": s.peak_year,
    "now": M(s.now), "last_season": s.last_season, "caps": s.caps.astype(int),
    "top5_18/19": s.m1819.astype(int), "top5_19/20": s.m1920.astype(int),
}).to_string(index=False))

print("\n=== ВЫБРАННЫЕ против ОСТАЛЬНЫХ молодых игроков (1500+ минут) ===")
rows = []
for lab, g in (("выбранные", o[o.sel == 1]), ("остальные", o[o.sel == 0])):
    rows.append({"группа": lab, "n": len(g), "возраст": round(g.age.mean(), 1),
                 "медиана старт, млн": round(g.start.median() / 1e6, 1),
                 "медиана 2019, млн": round(g.v2019.median() / 1e6, 1),
                 "пик ≥ 20 млн": f"{(g.peak >= 20e6).mean() * 100:.0f}%",
                 "пик ≥ 50 млн": f"{(g.peak >= 50e6).mean() * 100:.0f}%",
                 "топ-5 лиг 900+ мин позже": f"{g.top5_late.mean() * 100:.0f}%",
                 "хотя бы 1 матч за сборную": f"{(g.caps >= 1).mean() * 100:.0f}%",
                 "рост к 2019 (медиана)": f"{(np.exp(g.g19.median()) - 1) * 100:+.0f}%"})
print(pd.DataFrame(rows).to_string(index=False))

print("\nточный тест Фишера (выбранные против остальных):")
for lab, col in (("пик ≥ 20 млн", (o.peak >= 20e6)), ("топ-5 лиг позже", o.top5_late == 1),
                 ("есть матчи за сборную", o.caps >= 1)):
    tab = [[int(col[o.sel == 1].sum()), int((~col[o.sel == 1]).sum())],
           [int(col[o.sel == 0].sum()), int((~col[o.sel == 0]).sum())]]
    print(f"  {lab}: {tab[0]} против {tab[1]}, p={fisher_exact(tab)[1]:.3f}")

print("\nрегрессия: рост стоимости к 2019 с учётом стартовой цены, возраста и группы позиций")
sm_ = o[o.g19.notna() & o.start.notna()].copy()
sm_["logstart"] = np.log(sm_.start)
m = smf.ols("g19 ~ sel + logstart + age + C(group)", data=sm_).fit(cov_type="HC3")
lo, hi = m.conf_int().loc["sel"].values
pc = lambda x: (np.exp(x) - 1) * 100
print(f"n={len(sm_)}, выбранных {int(sm_.sel.sum())}: {pc(m.params['sel']):+.0f}% "
      f"(95% ДИ {pc(lo):+.0f}% … {pc(hi):+.0f}%), p={m.pvalues['sel']:.3f}")

o.to_csv(f"{folder}/u23_outcomes.csv", index=False)
print("\nсохранено: u23_outcomes.csv")

Mounted at /content/drive
молодых игроков (до 23, 1500+ минут): 101 | выбрано: 22 | без Transfermarkt-id: 0

=== ЧТО СТАЛО С ВЫБРАННЫМИ (млн евро; top5 = минуты в пяти топ-лигах в 2018/19 и 2019/20) ===
             player                   team role  age  comp  2016  2017  2018  2019  2021  peak  peak_y   now last_season  caps  top5_18/19  top5_19/20
        Keita Baldé                  Lazio AM_W 21.0  80.0  10.0  25.0  30.0  25.0   8.0  30.0    2018   0.9        2024    40         884         853
    Ousmane Dembélé                 Rennes AM_W 19.0  78.0  12.0  33.0  80.0 100.0  50.0 120.0    2019 100.0        2025    65        1671         358
              Bruma          Real Sociedad AM_W 22.0  74.0   5.0  11.0  12.5  10.0   5.0  12.5    2017   3.0        2025    12         549           0
    Raheem Sterling        Manchester City AM_W 22.0  73.0  45.0  45.0  90.0 140.0  90.0 160.0    2019   3.0        2025    82        2777        2205
    Anthony Martial      Manchester United

In [9]:
import glob, re
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

leagues = {"seriea": "Serie A", "premier": "Premier League",
           "laliga": "La Liga", "ligue1": "Ligue 1"}
ON_TARGET = ["Goal", "Saved", "Saved to Post"]


def load_events(key):
    files = glob.glob(f"{folder}/parts2/{key}_*.pkl")
    files = sorted(files, key=lambda p: int(re.search(r"_(\d+)\.pkl$", p).group(1)))
    return pd.concat([pd.read_pickle(p) for p in files], ignore_index=True)


def gk_league(key, name):
    ev = load_events(key)
    match_end = ev.groupby("match_id")["minute"].max()
    stint = {}
    for _, r in ev[ev.type == "Starting XI"].iterrows():
        for p in r.tactics["lineup"]:
            stint[(r.match_id, p["player"]["name"])] = [0, match_end[r.match_id], r.team]
    for _, r in ev[ev.type == "Substitution"].iterrows():
        mid = r.match_id
        if (mid, r.player) in stint:
            stint[(mid, r.player)][1] = r.minute
        stint[(mid, r.substitution_replacement)] = [r.minute, match_end[mid], r.team]
    st = pd.DataFrame(
        [(m, p, t, s, e) for (m, p), (s, e, t) in stint.items()],
        columns=["match_id", "player", "team", "start", "end"])
    st["match_end"] = st.match_id.map(match_end)

    named = ev.dropna(subset=["player", "position"])
    pos = named.groupby("player")["position"].agg(lambda s: s.mode().iat[0])
    gks = set(pos[pos == "Goalkeeper"].index)
    gst = st[st.player.isin(gks)].copy()
    gst["mins"] = gst.end - gst.start

    cols = ["match_id", "team", "minute", "shot_outcome", "shot_statsbomb_xg", "shot_type"]
    sh = ev.loc[ev.type == "Shot", cols].rename(columns={"team": "shooter"})
    sh = sh[sh.shot_type != "Penalty"]
    all_goals = int((sh.shot_outcome == "Goal").sum())
    keep = ["match_id", "player", "team", "start", "end", "match_end"]
    m = sh.merge(gst[keep], on="match_id")
    in_window = (m.minute >= m.start) & ((m.minute < m.end) | (m.end >= m.match_end))
    m = m[(m.shooter != m.team) & in_window].copy()
    m["on"] = m.shot_outcome.isin(ON_TARGET)
    m["goal"] = m.shot_outcome == "Goal"
    m["xg"] = m.shot_statsbomb_xg.fillna(0)
    agg = m.groupby("player").agg(shots=("xg", "size"), on_target=("on", "sum"),
                                  goals=("goal", "sum"), xga=("xg", "sum"))
    print(name, "голы привязаны / все:", int(agg.goals.sum()), "/", all_goals)

    pas = ev[(ev.type == "Pass") & ev.player.isin(gks)].copy()
    pas["done"] = pas.pass_outcome.isna()
    pas["long"] = pas.pass_length >= 35
    pas["long_done"] = pas["long"] & pas["done"]
    pg = pas.groupby("player").agg(passes=("done", "size"), passes_done=("done", "sum"),
                                   long_passes=("long", "sum"), long_done=("long_done", "sum"))

    gev = ev[ev.player.isin(gks) & (ev.type == "Goal Keeper")]
    gkt = gev.groupby(["player", "goalkeeper_type"]).size().unstack(fill_value=0)
    for c in ["Keeper Sweeper", "Collected", "Smother"]:
        if c not in gkt.columns:
            gkt[c] = 0
    gkt["sweeper"] = gkt["Keeper Sweeper"]
    gkt["claims"] = gkt["Collected"] + gkt["Smother"]

    mins = gst.groupby("player").mins.sum()
    team = gst.sort_values("mins", ascending=False).drop_duplicates("player")
    team = team.set_index("player").team
    out = pd.DataFrame({"mins": mins, "team": team})
    out = out.join(agg).join(pg).join(gkt[["sweeper", "claims"]])
    out["league"] = name
    return out


parts = [gk_league(k, n) for k, n in leagues.items()]
gk = pd.concat(parts)
gk.index.name = "player"
gk = gk.reset_index()
num = ["shots", "on_target", "goals", "xga", "passes", "passes_done",
       "long_passes", "long_done", "sweeper", "claims"]
for c in num:
    gk[c] = gk[c].fillna(0)
gk = gk[gk.mins >= 900].copy()

prior_save = (gk.on_target.sum() - gk.goals.sum()) / gk.on_target.sum()
prior_long = gk.long_done.sum() / gk.long_passes.sum()
gk["save_s"] = (gk.on_target - gk.goals + 20 * prior_save) / (gk.on_target + 20)
gk["xga_diff90"] = (gk.xga - gk.goals) / gk.mins * 90
gk["pass_pct"] = gk.passes_done / gk.passes.replace(0, np.nan)
gk["long_s"] = (gk.long_done + 20 * prior_long) / (gk.long_passes + 20)
gk["sweeper90"] = gk.sweeper / gk.mins * 90
gk["claims90"] = gk.claims / gk.mins * 90

SPEC = ["save_s", "xga_diff90", "pass_pct", "long_s", "sweeper90", "claims90"]
P = gk[SPEC].rank(pct=True)
gk["comp"] = P.mean(axis=1)
gk["n_high"] = (P >= 0.8).sum(axis=1)

gk.to_pickle(f"{folder}/gk_stats.pkl")
print("вратарей с 900+ минутами:", len(gk))
print("сохранено: gk_stats.pkl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Serie A голы привязаны / все: 861 / 858
Premier League голы привязаны / все: 914 / 914
La Liga голы привязаны / все: 956 / 945
Ligue 1 голы привязаны / все: 855 / 852
вратарей с 900+ минутами: 101
сохранено: gk_stats.pkl


In [10]:
import numpy as np
import pandas as pd
import duckdb
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
con = duckdb.connect(f"{folder}/transfermarkt-datasets.duckdb", read_only=True)

comp_of = {"Serie A": "IT1", "Premier League": "GB1", "La Liga": "ES1", "Ligue 1": "FR1"}
gk = pd.read_pickle(f"{folder}/gk_stats.pkl")

mer = pd.read_pickle(f"{folder}/all_1516_merged.pkl")
team_map = mer.dropna(subset=["tm_club"]).groupby("team").tm_club
team_map = team_map.agg(lambda s: s.mode().iat[0]).to_dict()

raw = con.sql("""
    SELECT v.player_id AS tm_id, v.date AS tm_date, v.market_value_in_eur AS value_eur,
           v.current_club_name AS tm_club, p.name AS tm_name, p.date_of_birth
    FROM player_valuations v JOIN players p USING (player_id)
    WHERE p.position = 'Goalkeeper'
      AND v.date >= DATE '2015-07-01' AND v.date <= DATE '2016-09-30'
""").df()
raw["tm_date"] = pd.to_datetime(raw.tm_date)
raw = raw[raw.value_eur.notna() & (raw.value_eur > 0)]
cut = pd.Timestamp("2016-06-30")
b4 = raw[raw.tm_date <= cut].sort_values("tm_date").groupby("tm_id").tail(1)
rest = raw[(raw.tm_date > cut) & ~raw.tm_id.isin(b4.tm_id)]
af = rest.sort_values("tm_date").groupby("tm_id").head(1)
gv = pd.concat([b4, af], ignore_index=True)
age_days = (cut - pd.to_datetime(gv.date_of_birth)).dt.days
gv["age"] = (age_days / 365.25).round(1)

tm_min = con.sql("""
    SELECT player_id AS tm_id, competition_id AS comp, SUM(minutes_played) AS tm_mins
    FROM appearances
    WHERE date >= DATE '2015-07-01' AND date <= DATE '2016-06-30'
      AND competition_id IN ('IT1','GB1','ES1','FR1')
    GROUP BY player_id, competition_id
""").df()
mins_lookup = {(int(r.tm_id), r.comp): r.tm_mins for r in tm_min.itertuples()}

rows = []
for i, r in gk.iterrows():
    club = team_map.get(r.team)
    pool = gv[gv.tm_club == club] if club else gv.iloc[0:0]
    cands = []
    for c in pool.itertuples():
        mt = mins_lookup.get((int(c.tm_id), comp_of[r.league]))
        if mt is not None and 0.85 <= mt / r.mins <= 1.10:
            cands.append((abs(mt / r.mins - 0.963), c.tm_id, mt / r.mins))
    cands.sort()
    how = "unique" if len(cands) == 1 else ("closest" if cands else "unmatched")
    tid = cands[0][1] if cands else np.nan
    rat = cands[0][2] if cands else np.nan
    rows.append((i, how, tid, rat))

lk = pd.DataFrame(rows, columns=["i", "how", "tm_id", "ratio"]).set_index("i")
gk = gk.join(lk)
gk = gk.merge(gv[["tm_id", "tm_name", "tm_club", "value_eur", "age"]], on="tm_id", how="left")
gk["name"] = gk.tm_name.fillna(gk.player)

linked = int(gk.tm_id.notna().sum())
big = int((gk.mins >= 1500).sum())
print("вратарей:", len(gk), "| связано с Transfermarkt:", linked, "| с 1500+ минутами:", big)
print("тип связи:", gk.how.value_counts().to_dict())
missing = gk[gk.tm_id.isna() & (gk.mins >= 1500)]
print("несвязанные с 1500+ минутами:", missing.player.tolist())

cols = ["name", "team", "age", "mins", "save_s", "xga_diff90",
        "pass_pct", "long_s", "sweeper90", "claims90", "comp"]


def show(d):
    return d[cols].round(2).to_string(index=False)


print("\n=== лучшие 8 вратарей всех возрастов (проверка здравого смысла) ===")
top = gk[gk.mins >= 1500].sort_values("comp", ascending=False).head(8)
print(show(top))
print("\n=== вратари до 23 лет с 1500+ минутами ===")
young = gk[(gk.age < 23) & (gk.mins >= 1500)].sort_values("comp", ascending=False)
print(show(young))

gk.to_pickle(f"{folder}/gk_1516_scored.pkl")
print("\nсохранено: gk_1516_scored.pkl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
вратарей: 101 | связано с Transfermarkt: 99 | с 1500+ минутами: 83
тип связи: {'unique': 92, 'closest': 7, 'unmatched': 2}
несвязанные с 1500+ минутами: []

=== лучшие 8 вратарей всех возрастов (проверка здравого смысла) ===
               name                team  age  mins  save_s  xga_diff90  pass_pct  long_s  sweeper90  claims90  comp
      Carlos Kameni              Málaga 32.4  2550    0.77        0.29      0.66    0.51       1.09      2.22  0.89
Alexandre Letellier              Angers 25.6  1772    0.77        0.30      0.62    0.48       1.47      1.37  0.85
  Wojciech Szczesny             AS Roma 26.2  3184    0.78        0.14      0.72    0.44       1.27      1.02  0.77
   Samir Handanovic         Inter Milan 32.0  3377    0.79        0.25      0.75    0.50       1.09      0.56  0.76
       Keylor Navas         Real Madrid 29.5  3152    0.76        

In [11]:
import numpy as np
import pandas as pd
import duckdb
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
con = duckdb.connect(f"{folder}/transfermarkt-datasets.duckdb", read_only=True)

gk = pd.read_pickle(f"{folder}/gk_1516_scored.pkl")
u = gk[(gk.age < 23) & (gk.mins >= 1500)].sort_values("comp", ascending=False).copy()
ids = ",".join(str(int(i)) for i in u.tm_id.unique())

pl = con.sql(f"""
    SELECT player_id AS tm_id, last_season, international_caps AS caps
    FROM players WHERE player_id IN ({ids})
""").df().set_index("tm_id")
val = con.sql(f"""
    SELECT player_id AS tm_id, date, market_value_in_eur AS v
    FROM player_valuations
    WHERE player_id IN ({ids}) AND market_value_in_eur IS NOT NULL
""").df()
val["date"] = pd.to_datetime(val.date)
app = con.sql(f"""
    SELECT player_id AS tm_id, date, minutes_played AS mp FROM appearances
    WHERE player_id IN ({ids})
      AND competition_id IN ('GB1','ES1','IT1','FR1','L1')
      AND date >= DATE '2016-07-01'
""").df()
app["date"] = pd.to_datetime(app.date)


def at(y):
    t = pd.Timestamp(f"{y}-06-30")
    lo, hi = t - pd.Timedelta(days=150), t + pd.Timedelta(days=30)
    w = val[(val.date >= lo) & (val.date <= hi)].copy()
    w["gap"] = (w.date - t).abs()
    return w.sort_values("gap").groupby("tm_id").head(1).set_index("tm_id").v


def season_mins(y):
    a, e = pd.Timestamp(f"{y}-07-01"), pd.Timestamp(f"{y + 1}-06-30")
    return app[(app.date >= a) & (app.date <= e)].groupby("tm_id").mp.sum()


def M(x):
    return np.round(x / 1e6, 1)


pk = val.sort_values(["tm_id", "v", "date"], ascending=[True, False, True])
pk = pk.groupby("tm_id").head(1).set_index("tm_id")

u["rank"] = range(1, len(u) + 1)
for y in (2017, 2018, 2019, 2021):
    u[f"v{y}"] = u.tm_id.map(at(y))

res = pd.DataFrame({
    "rank": u["rank"], "player": u.name, "team": u.team, "age": u.age,
    "comp": (u.comp * 100).round(0),
    "2016": M(u.value_eur), "2017": M(u.v2017), "2018": M(u.v2018),
    "2019": M(u.v2019), "2021": M(u.v2021),
    "peak": M(u.tm_id.map(pk.v)), "peak_y": u.tm_id.map(pk.date.dt.year),
    "last_season": u.tm_id.map(pl.last_season),
    "caps": u.tm_id.map(pl.caps).fillna(0).astype(int),
    "top5_18/19": u.tm_id.map(season_mins(2018)).fillna(0).astype(int),
    "top5_19/20": u.tm_id.map(season_mins(2019)).fillna(0).astype(int),
})
print("=== ЧТО СТАЛО С МОЛОДЫМИ ВРАТАРЯМИ (rank 1-3 = выбранные по правилу; млн евро) ===")
print(res.to_string(index=False))
res.to_csv(f"{folder}/gk_u23_outcomes.csv", index=False)
print("\nсохранено: gk_u23_outcomes.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== ЧТО СТАЛО С МОЛОДЫМИ ВРАТАРЯМИ (rank 1-3 = выбранные по правилу; млн евро) ===
 rank               player          team  age  comp  2016  2017  2018  2019  2021  peak  peak_y last_season  caps  top5_18/19  top5_19/20
    1       Yoan Cardinale      OGC Nice 22.3  62.0   0.6   5.5   3.0   2.0   0.6   5.5    2017        2020     0         270           0
    2          Sergio Rico       Sevilla 22.8  61.0  15.0  20.0  12.0  12.0   7.0  20.0    2017        2023     0        2610         180
    3 Gianluigi Donnarumma      AC Milan 17.3  58.0   8.0  25.0  40.0  55.0  65.0  65.0    2021        2025    83        3092        2302
    4    Pierluigi Gollini Hellas Verona 21.3  41.0   2.0   4.0   3.5   6.0  17.0  17.0    2020        2025     1        1800        2250
    5      Paul Bernardoni      Bordeaux 19.2  36.0   1.2   0.8   2.0   9.0  10.0  10.0    2021   